In [4]:
# Ejemplo de la página 88 del libro "Deep Learning with Python Second Edition" de François Chollet

import tensorflow as tf

class NaiveDense:
    def __init__(self, input_size, output_size, activation):
        self.activation = activation
        w_shape = (input_size, output_size)
        w_initial_value = tf.random.uniform(w_shape, minval=0, maxval=1e-1)
        self.W = tf.Variable(w_initial_value)
        b_shape = (output_size,)
        b_initial_value = tf.zeros(b_shape)
        self.b = tf.Variable(b_initial_value)

    def __call__(self, inputs):
        return self.activation(tf.matmul(inputs, self.W) + self.b)

    @property
    def weights(self):
        #"""Devuelve lista de pesos entrenables [W, b]"""
        return [self.W, self.b]

In [2]:
# Ejemplo de uso del código anterior por DeepSeek:

# ========== EJEMPLO DE USO ==========
if __name__ == "__main__":
    # Crear capa
    layer = NaiveDense(input_size=3, output_size=2, activation=tf.nn.relu)
    
    # Datos de entrada (batch_size=4, features=3)
    x = tf.constant([[1.0, 2.0, 3.0],
                     [4.0, 5.0, 6.0],
                     [7.0, 8.0, 9.0],
                     [10.0, 11.0, 12.0]], dtype=tf.float32)
    
    # Forward pass
    output = layer(x)  # ¡Ahora funciona!
    print("Entrada (shape: {}):".format(x.shape))
    print(x.numpy())
    print("\nPesos W (shape: {}):".format(layer.W.shape))
    print(layer.W.numpy())
    print("\nBiases b (shape: {}):".format(layer.b.shape))
    print(layer.b.numpy())
    print("\nSalida (shape: {}):".format(output.shape))
    print(output.numpy())
    print("\nLista de pesos (property):", layer.weights)

Entrada (shape: (4, 3)):
[[ 1.  2.  3.]
 [ 4.  5.  6.]
 [ 7.  8.  9.]
 [10. 11. 12.]]

Pesos W (shape: (3, 2)):
[[0.09302709 0.05427637]
 [0.01563764 0.09687904]
 [0.07781436 0.05074089]]

Biases b (shape: (2,)):
[0. 0.]

Salida (shape: (4, 2)):
[[0.35774544 0.4002571 ]
 [0.9171827  1.005946  ]
 [1.47662    1.611635  ]
 [2.0360572  2.2173238 ]]

Lista de pesos (property): [<tf.Variable 'Variable:0' shape=(3, 2) dtype=float32, numpy=
array([[0.09302709, 0.05427637],
       [0.01563764, 0.09687904],
       [0.07781436, 0.05074089]], dtype=float32)>, <tf.Variable 'Variable:0' shape=(2,) dtype=float32, numpy=array([0., 0.], dtype=float32)>]


In [5]:
# Ejemplo de la página 89 del libro "Deep Learning with Python Second Edition" de François Chollet
# Creamos una clase secuencial simple que encadena varias capas NaiveDense
class NaiveSequential:

    def __init__(self, layers):
        self.layers = layers
    def __call__(self, inputs):
        x = inputs
        for layer in self.layers:
            x = layer(x)
        return x
    
    @property
    def weights(self):
        weights = []
        for layer in self.layers:
            weights += layer.weights
        return weights

# Usando la Clase NaiveDense y NaiveSequential creamos un modelo Keras simulado:

model = NaiveSequential([
    NaiveDense(input_size=28 * 28, output_size=512, activation=tf.nn.relu),
    NaiveDense(input_size=512, output_size=10, activation=tf.nn.softmax)
    ])
# Verificamos que el modelo tiene 4 conjuntos de pesos (W y b para cada capa)
assert len(model.weights) == 4

In [6]:
# Creando un Batch Generator (Generador de Lotes) simple

import math

class BatchGenerator:

    def __init__(self, images, labels, batch_size=128):
        assert len(images) == len(labels)
        self.index = 0
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        self.num_batches = math.ceil(len(images) / batch_size)

    def next(self):
        images = self.images[self.index : self.index + self.batch_size]
        labels = self.labels[self.index : self.index + self.batch_size]
        self.index += self.batch_size
        return images, labels

In [7]:
# Cálculo del Gradiente:

def one_training_step(model, images_batch, labels_batch):
    with tf.GradientTape() as tape:
        # Cálculo del "Fordward pass" según el modelo GradientTape de Tensor Flow
        predictions = model(images_batch)
        per_sample_losses = tf.keras.losses.sparse_categorical_crossentropy(
            labels_batch, predictions)
        average_loss = tf.reduce_mean(per_sample_losses)
        # ---------------------------------------------------------------------------
    # Cálculo del gradiente con respecto a los pesos del modelo. 
    # La salida "gradients" es una lista donde cada entrada corresponde a un peso de la 
    # lista model.weights
    gradients = tape.gradient(average_loss, model.weights)
    # ---------------------------------------------------------------------------------
    # Actualizamos los pesos usando los gradientes calculados.
    update_weights(gradients, model.weights) # Vamos a definir esta función a continuación
    return average_loss

In [ ]:
# Función para actuallizar los pesos del modelo usando los gradientes calculados
# (Este código no se usa pero se usa el Optimizador de Keras)

learning_rate = 1e-3

def update_weights(gradients, weights):
    for g, w in zip(gradients, weights):
        w.assign_sub(g * learning_rate) # assign_sub es equivalente a w = w - g*learning_rate.

In [8]:
# Optimizador de Keras equivalente a la función anterior

from tensorflow.keras import optimizers

optimizer = optimizers.SGD(learning_rate=1e-3)

def update_weights(gradients, weights):
    optimizer.apply_gradients(zip(gradients, weights))

In [10]:
# Ciclo de entrenamiento completo - Páginas 90 y 91 del libro

def fit(model, images, labels, epochs, batch_size=128):
    for epoch_counter in range(epochs):
        print(f"Epoch {epoch_counter}")
        batch_generator = BatchGenerator(images, labels)
        for batch_counter in range(batch_generator.num_batches):
            images_batch, labels_batch = batch_generator.next()
            loss = one_training_step(model, images_batch, labels_batch)
            if batch_counter % 100 == 0:
                print(f"loss at batch {batch_counter}: {loss:.2f}")

# Cargando datos de MNIST

from tensorflow.keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype("float32") / 255

fit(model, train_images, train_labels, epochs=10, batch_size=128)

Epoch 0
loss at batch 0: 0.64
loss at batch 100: 0.66
loss at batch 200: 0.57
loss at batch 300: 0.64
loss at batch 400: 0.70
Epoch 1
loss at batch 0: 0.61
loss at batch 100: 0.62
loss at batch 200: 0.54
loss at batch 300: 0.61
loss at batch 400: 0.68
Epoch 2
loss at batch 0: 0.58
loss at batch 100: 0.59
loss at batch 200: 0.51
loss at batch 300: 0.59
loss at batch 400: 0.65
Epoch 3
loss at batch 0: 0.56
loss at batch 100: 0.56
loss at batch 200: 0.48
loss at batch 300: 0.56
loss at batch 400: 0.63
Epoch 4
loss at batch 0: 0.54
loss at batch 100: 0.54
loss at batch 200: 0.46
loss at batch 300: 0.54
loss at batch 400: 0.62
Epoch 5
loss at batch 0: 0.52
loss at batch 100: 0.52
loss at batch 200: 0.45
loss at batch 300: 0.53
loss at batch 400: 0.60
Epoch 6
loss at batch 0: 0.50
loss at batch 100: 0.50
loss at batch 200: 0.43
loss at batch 300: 0.51
loss at batch 400: 0.59
Epoch 7
loss at batch 0: 0.49
loss at batch 100: 0.48
loss at batch 200: 0.42
loss at batch 300: 0.50
loss at batch 40

In [11]:
# Prueba completa del código con datos de MNIST - Página 91 del libro
# La prueba del libro supone que numpy ya está instalado pero en mi VSCode no lo estaba

import numpy as np

predictions = model(test_images)
predictions = predictions.numpy()
predicted_labels = np.argmax(predictions, axis=1)
matches = predicted_labels == test_labels
print(f"accuracy: {matches.mean():.2f}")

accuracy: 0.86
